# Text Feature Engineering for Price Prediction

This notebook implements **advanced text feature extraction** to dramatically improve price prediction accuracy.

In [ ]:
import re
import pickle
import logging
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

# Text processing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Sentence transformers for semantic embeddings
from sentence_transformers import SentenceTransformer

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Paths
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "dataset"
FEATURES_DIR = PROJECT_DIR / "features"
FEATURES_DIR.mkdir(exist_ok=True)

## Load Data

In [4]:
# Load the datasets
print("Loading datasets...")
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

# Fill missing text
df_train['catalog_content'] = df_train['catalog_content'].fillna('')
df_test['catalog_content'] = df_test['catalog_content'].fillna('')

print("\nSample catalog content:")
print(df_train['catalog_content'].iloc[0][:300])

Loading datasets...
Train shape: (75000, 4)
Test shape: (75000, 3)

Columns: ['sample_id', 'catalog_content', 'image_link', 'price']

Sample catalog content:
Item Name: La Victoria Green Taco Sauce Mild, 12 Ounce (Pack of 6)
Value: 72.0
Unit: Fl Oz



## 1. Numeric Feature Extraction

E-commerce prices are heavily influenced by physical attributes. Extract:
- Weight (oz, lb, kg, g)
- Volume (fl oz, ml, L, gallons)
- Dimensions (inches, cm)
- Pack/Count quantities
- Model numbers and specifications

In [5]:
def extract_numeric_features(text: str) -> Dict[str, float]:
    """
    Extract numeric features that strongly correlate with price.
    These are domain-specific signals that TF-IDF completely misses.
    """
    if not isinstance(text, str) or not text.strip():
        return {
            'weight_oz': 0.0,
            'volume_oz': 0.0,
            'dimension_inches': 0.0,
            'pack_count': 1.0,
            'has_weight': 0,
            'has_volume': 0,
            'has_dimensions': 0,
            'numeric_spec_count': 0,
        }

    text_lower = text.lower()

    # Weight extraction (standardize to ounces)
    weight_oz = 0.0
    has_weight = 0

    # Pounds
    lb_match = re.search(r'(\d+\.?\d*)\s*(?:lb|lbs|pound|pounds)\b', text_lower)
    if lb_match:
        weight_oz = float(lb_match.group(1)) * 16
        has_weight = 1

    # Ounces
    oz_match = re.search(r'(\d+\.?\d*)\s*(?:oz|ounce|ounces)\b(?!\s*fl)', text_lower)
    if oz_match:
        weight_oz = max(weight_oz, float(oz_match.group(1)))
        has_weight = 1

    # Kilograms
    kg_match = re.search(r'(\d+\.?\d*)\s*(?:kg|kilogram|kilograms)\b', text_lower)
    if kg_match:
        weight_oz = max(weight_oz, float(kg_match.group(1)) * 35.274)
        has_weight = 1

    # Grams
    g_match = re.search(r'(\d+\.?\d*)\s*(?:g|gram|grams)\b', text_lower)
    if g_match and not kg_match:  # Avoid confusion with kg
        weight_oz = max(weight_oz, float(g_match.group(1)) * 0.035274)
        has_weight = 1

    # Volume extraction (standardize to fluid ounces)
    volume_oz = 0.0
    has_volume = 0

    # Fluid ounces
    floz_match = re.search(r'(\d+\.?\d*)\s*(?:fl\s*oz|fluid\s*ounce)s?\b', text_lower)
    if floz_match:
        volume_oz = float(floz_match.group(1))
        has_volume = 1

    # Milliliters
    ml_match = re.search(r'(\d+\.?\d*)\s*(?:ml|milliliter)s?\b', text_lower)
    if ml_match:
        volume_oz = max(volume_oz, float(ml_match.group(1)) * 0.033814)
        has_volume = 1

    # Liters
    l_match = re.search(r'(\d+\.?\d*)\s*(?:l|liter|litre)s?\b', text_lower)
    if l_match and not ml_match:  # Avoid confusion with ml
        volume_oz = max(volume_oz, float(l_match.group(1)) * 33.814)
        has_volume = 1

    # Gallons
    gal_match = re.search(r'(\d+\.?\d*)\s*(?:gal|gallon)s?\b', text_lower)
    if gal_match:
        volume_oz = max(volume_oz, float(gal_match.group(1)) * 128)
        has_volume = 1

    # Dimensions (extract and compute volume in cubic inches)
    dimension_inches = 0.0
    has_dimensions = 0

    # Pattern: 10 x 5 x 3 inches or 10"x5"x3"
    dim_match = re.search(r'(\d+\.?\d*)\s*[x×]\s*(\d+\.?\d*)\s*[x×]\s*(\d+\.?\d*)\s*(?:inch|in|")?', text_lower)
    if dim_match:
        dims = [float(dim_match.group(i)) for i in range(1, 4)]
        dimension_inches = np.prod(dims)  # cubic inches
        has_dimensions = 1

    # Pack count extraction
    pack_count = 1.0
    pack_patterns = [
        r'pack\s*of\s*(\d+)',
        r'(\d+)\s*pack',
        r'(\d+)\s*count',
        r'(\d+)\s*piece',
        r'set\s*of\s*(\d+)',
        r'(\d+)\s*per\s*(?:case|box|pack)'
    ]

    for pattern in pack_patterns:
        match = re.search(pattern, text_lower)
        if match:
            count = int(match.group(1))
            if 1 < count <= 100:  # Reasonable range
                pack_count = float(count)
                break

    # Count numeric specifications (model numbers, specs)
    # These indicate more detailed/technical products which often cost more
    numeric_specs = re.findall(r'\d+(?:\.\d+)?(?:\s*(?:mhz|ghz|gb|mb|tb|mp|watts?|v|volts?)\b)', text_lower)
    numeric_spec_count = len(numeric_specs)

    return {
        'weight_oz': weight_oz,
        'volume_oz': volume_oz,
        'dimension_inches': dimension_inches,
        'pack_count': pack_count,
        'has_weight': has_weight,
        'has_volume': has_volume,
        'has_dimensions': has_dimensions,
        'numeric_spec_count': numeric_spec_count,
        'weight_log': np.log1p(weight_oz),
        'volume_log': np.log1p(volume_oz),
        'dimension_log': np.log1p(dimension_inches),
        'pack_log': np.log1p(pack_count),
    }


# Extract numeric features
print("Extracting numeric features from text...")
train_numeric_features = pd.DataFrame(
    df_train['catalog_content'].progress_apply(extract_numeric_features).tolist()
)
test_numeric_features = pd.DataFrame(
    df_test['catalog_content'].progress_apply(extract_numeric_features).tolist()
)

print(f"\nNumeric features extracted: {train_numeric_features.shape[1]} features")
print("\nFeature statistics (train):")
print(train_numeric_features.describe())

# Check correlation with price
if 'price' in df_train.columns:
    print("\n📊 Correlation with price:")
    correlations = pd.DataFrame({
        'feature': train_numeric_features.columns,
        'correlation': [train_numeric_features[col].corr(df_train['price']) for col in train_numeric_features.columns]
    }).sort_values('correlation', key=abs, ascending=False)
    print(correlations)

Extracting numeric features from text...


100%|██████████| 75000/75000 [00:23<00:00, 3251.95it/s]



Numeric features extracted: 12 features

Feature statistics (train):
          weight_oz      volume_oz  dimension_inches    pack_count  \
count  75000.000000   75000.000000      75000.000000  75000.000000   
mean      18.439412      18.592371          3.088618      5.433893   
std      162.662641    2220.689436        608.616795     11.768324   
min        0.000000       0.000000          0.000000      1.000000   
25%        0.000000       0.000000          0.000000      1.000000   
50%        5.000000       0.000000          0.000000      1.000000   
75%       15.500000       0.000000          0.000000      6.000000   
max    19200.000000  588363.600000     165888.000000    100.000000   

         has_weight    has_volume  has_dimensions  numeric_spec_count  \
count  75000.000000  75000.000000    75000.000000        75000.000000   
mean       0.715947      0.119573        0.001920            0.000573   
std        0.450965      0.324464        0.043776            0.041789   
min    

## 2. Brand and Category Intelligence

Extract brand signals and quality indicators from text.

In [ ]:
# Premium brand keywords
PREMIUM_KEYWORDS = [
    'premium', 'luxury', 'professional', 'deluxe', 'pro', 'ultra',
    'platinum', 'gold', 'elite', 'supreme', 'ultimate', 'signature',
    'organic', 'natural', 'handmade', 'artisan', 'gourmet',
    'imported', 'certified', 'authentic', 'original'
]

# Budget/value keywords
BUDGET_KEYWORDS = [
    'value', 'economy', 'budget', 'basic', 'standard', 'generic',
    'discount', 'affordable', 'bulk', 'wholesale'
]

# Material quality indicators 
PREMIUM_MATERIALS = [
    'stainless steel', 'titanium', 'aluminum', 'glass', 'ceramic',
    'leather', 'wood', 'bamboo', 'cotton', 'silk', 'wool',
    'brass', 'copper', 'bronze'
]

BUDGET_MATERIALS = [
    'plastic', 'pvc', 'vinyl', 'synthetic', 'polyester', 'nylon'
]


def extract_brand_quality_features(text: str) -> Dict[str, float]:
    """
    Extract brand positioning and quality signals.
    These help the model understand price tier.
    """
    if not isinstance(text, str):
        text = ''

    text_lower = text.lower()

    # Count premium and budget keywords
    premium_count = sum(1 for kw in PREMIUM_KEYWORDS if kw in text_lower)
    budget_count = sum(1 for kw in BUDGET_KEYWORDS if kw in text_lower)

    premium_material_count = sum(1 for mat in PREMIUM_MATERIALS if mat in text_lower)
    budget_material_count = sum(1 for mat in BUDGET_MATERIALS if mat in text_lower)

    # Extract brand name (first capitalized word or first word after "Item Name:")
    brand_match = re.search(r'Item Name:\s*([A-Z][a-z]+)', text)
    has_brand = 1 if brand_match else 0

    # Check for certifications and guarantees
    has_certification = int(any(cert in text_lower for cert in
                                ['certified', 'approved', 'rated', 'tested', 'verified']))
    has_warranty = int(any(war in text_lower for war in
                           ['warranty', 'guarantee', 'guaranteed']))

    # Check for new/refurbished/used
    is_new = int('new' in text_lower and 'brand new' in text_lower)
    is_refurbished = int(any(ref in text_lower for ref in ['refurbished', 'renewed', 'remanufactured']))

    return {
        'premium_keyword_count': premium_count,
        'budget_keyword_count': budget_count,
        'premium_material_count': premium_material_count,
        'budget_material_count': budget_material_count,
        'quality_score': premium_count + premium_material_count - budget_count - budget_material_count,
        'has_brand_name': has_brand,
        'has_certification': has_certification,
        'has_warranty': has_warranty,
        'is_new': is_new,
        'is_refurbished': is_refurbished,
    }


print("Extracting brand and quality features...")
train_brand_features = pd.DataFrame(
    df_train['catalog_content'].progress_apply(extract_brand_quality_features).tolist()
)
test_brand_features = pd.DataFrame(
    df_test['catalog_content'].progress_apply(extract_brand_quality_features).tolist()
)

print(f"\nBrand features extracted: {train_brand_features.shape[1]} features")
print("\nFeature statistics (train):")
print(train_brand_features.describe())

if 'price' in df_train.columns:
    print("\n📊 Correlation with price:")
    correlations = pd.DataFrame({
        'feature': train_brand_features.columns,
        'correlation': [train_brand_features[col].corr(df_train['price']) for col in train_brand_features.columns]
    }).sort_values('correlation', key=abs, ascending=False)
    print(correlations)

Extracting brand and quality features...


100%|██████████| 75000/75000 [00:05<00:00, 14154.48it/s]



Brand features extracted: 10 features

Feature statistics (train):
       premium_keyword_count  budget_keyword_count  premium_material_count  \
count           75000.000000          75000.000000            75000.000000   
mean                1.686733              1.182333                0.072893   
std                 1.493278              0.474915                0.277526   
min                 0.000000              1.000000                0.000000   
25%                 1.000000              1.000000                0.000000   
50%                 1.000000              1.000000                0.000000   
75%                 3.000000              1.000000                0.000000   
max                10.000000              5.000000                5.000000   

       budget_material_count  quality_score  has_brand_name  \
count           75000.000000   75000.000000    75000.000000   
mean                0.034440       0.542853        0.877573   
std                 0.186907       1.460

## 3. Sentence Transformer Embeddings

**This is the game-changer!** Use pre-trained transformer models to get semantic embeddings.
These capture meaning far better than TF-IDF.

Models to try (in order of effectiveness):
- `paraphrase-mpnet-base-v2` (768 dim, best quality)
- `all-MiniLM-L6-v2` (384 dim, fast & good)
- `all-mpnet-base-v2` (768 dim, excellent)

In [7]:
if SENTENCE_TRANSFORMER_AVAILABLE:
    # Use a smaller, faster model that still performs excellently
    MODEL_NAME = 'all-MiniLM-L6-v2'  # 384 dimensions, fast
    # MODEL_NAME = 'paraphrase-mpnet-base-v2'  # 768 dimensions, slower but better

    print(f"Loading Sentence Transformer model: {MODEL_NAME}")
    print("This will download ~80MB on first run...")

    sentence_model = SentenceTransformer(MODEL_NAME)

    # Clean text for embedding
    def clean_for_embedding(text: str) -> str:
        if not isinstance(text, str):
            return ""
        # Remove HTML, excessive whitespace
        text = re.sub(r'<[^>]+>', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        # Truncate to first 512 tokens (model limit)
        words = text.split()[:512]
        return ' '.join(words).strip()

    print("\nPreparing text for embedding...")
    train_texts = df_train['catalog_content'].apply(clean_for_embedding).tolist()
    test_texts = df_test['catalog_content'].apply(clean_for_embedding).tolist()

    # Generate embeddings in batches (faster)
    print("\nGenerating train embeddings...")
    print("This may take 5-10 minutes for 75k samples...")
    train_embeddings = sentence_model.encode(
        train_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    print("\nGenerating test embeddings...")
    test_embeddings = sentence_model.encode(
        test_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    print(f"\n✓ Train embeddings shape: {train_embeddings.shape}")
    print(f"✓ Test embeddings shape: {test_embeddings.shape}")

    # Save embeddings
    np.save(FEATURES_DIR / 'sentence_embeddings_train.npy', train_embeddings)
    np.save(FEATURES_DIR / 'sentence_embeddings_test.npy', test_embeddings)
    print(f"\n✓ Saved embeddings to {FEATURES_DIR}")

else:
    print("⚠️ Sentence Transformers not available.")
    print("Install with: pip install sentence-transformers")
    print("\nFor now, generating placeholder embeddings...")

    # Create empty placeholders
    train_embeddings = np.zeros((len(df_train), 384), dtype=np.float32)
    test_embeddings = np.zeros((len(df_test), 384), dtype=np.float32)

Loading Sentence Transformer model: all-MiniLM-L6-v2
This will download ~80MB on first run...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...


Preparing text for embedding...

Generating train embeddings...
This may take 5-10 minutes for 75k samples...


Batches:   0%|          | 0/1172 [00:00<?, ?it/s]


Generating test embeddings...


Batches:   0%|          | 0/1172 [00:00<?, ?it/s]


✓ Train embeddings shape: (75000, 384)
✓ Test embeddings shape: (75000, 384)

✓ Saved embeddings to /content/features


## 4. Enhanced TF-IDF with Character N-grams

Character n-grams help capture brand names, product codes, and model numbers that word-level TF-IDF misses.

In [ ]:
print("Creating enhanced TF-IDF with character n-grams...")

# Clean text for TF-IDF
def clean_for_tfidf(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.strip()

train_clean_text = df_train['catalog_content'].apply(clean_for_tfidf)
test_clean_text = df_test['catalog_content'].apply(clean_for_tfidf)

# Word-level TF-IDF with better parameters
print("\nFitting word-level TF-IDF...")
word_vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 3),  # Unigrams, bigrams, trigrams
    min_df=3,
    max_df=0.90,
    sublinear_tf=True,
    strip_accents='unicode'
)

word_tfidf_train = word_vectorizer.fit_transform(train_clean_text)
word_tfidf_test = word_vectorizer.transform(test_clean_text)

print(f"Word TF-IDF shape: {word_tfidf_train.shape}")

# Character-level TF-IDF (captures brand names and codes)
print("\nFitting character-level TF-IDF...")
char_vectorizer = TfidfVectorizer(
    max_features=1000,
    analyzer='char',
    ngram_range=(3, 5),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)

char_tfidf_train = char_vectorizer.fit_transform(train_clean_text)
char_tfidf_test = char_vectorizer.transform(test_clean_text)

print(f"Character TF-IDF shape: {char_tfidf_train.shape}")
print("\nApplying SVD to reduce dimensions...")
n_components_word = 300
n_components_char = 100

svd_word = TruncatedSVD(n_components=n_components_word, random_state=42)
word_svd_train = svd_word.fit_transform(word_tfidf_train)
word_svd_test = svd_word.transform(word_tfidf_test)

svd_char = TruncatedSVD(n_components=n_components_char, random_state=42)
char_svd_train = svd_char.fit_transform(char_tfidf_train)
char_svd_test = svd_char.transform(char_tfidf_test)

print(f"\nWord SVD variance explained: {svd_word.explained_variance_ratio_.sum():.3f}")
print(f"Character SVD variance explained: {svd_char.explained_variance_ratio_.sum():.3f}")

print(f"\nFinal shapes:")
print(f"  Word SVD: {word_svd_train.shape}")
print(f"  Char SVD: {char_svd_train.shape}")

Creating enhanced TF-IDF with character n-grams...

Fitting word-level TF-IDF...
Word TF-IDF shape: (75000, 3000)

Fitting character-level TF-IDF...
Character TF-IDF shape: (75000, 1000)

Applying SVD to reduce dimensions...

Word SVD variance explained: 0.553
Character SVD variance explained: 0.706

Final shapes:
  Word SVD: (75000, 300)
  Char SVD: (75000, 100)


## 5. Combine All Enhanced Text Features

In [ ]:
print("Combining all enhanced text features...")

# Combine numeric and brand features
train_handcrafted = np.concatenate([
    train_numeric_features.values,
    train_brand_features.values
], axis=1)

test_handcrafted = np.concatenate([
    test_numeric_features.values,
    test_brand_features.values
], axis=1)

# Combine with TF-IDF features
train_tfidf_combined = np.concatenate([
    word_svd_train,
    char_svd_train
], axis=1)

test_tfidf_combined = np.concatenate([
    word_svd_test,
    char_svd_test
], axis=1)

train_text_features_enhanced = np.concatenate([
    train_handcrafted,
    train_embeddings,
    train_tfidf_combined
], axis=1)

test_text_features_enhanced = np.concatenate([
    test_handcrafted,
    test_embeddings,
    test_tfidf_combined
], axis=1)

print(f"\nEnhanced text features shape:")
print(f"  Train: {train_text_features_enhanced.shape}")
print(f"  Test: {test_text_features_enhanced.shape}")

print(f"\nFeature breakdown:")
print(f"  - Numeric features: {train_numeric_features.shape[1]}")
print(f"  - Brand/quality features: {train_brand_features.shape[1]}")
if SENTENCE_TRANSFORMER_AVAILABLE:
    print(f"  - Sentence embeddings: {train_embeddings.shape[1]}")
print(f"  - Word TF-IDF (SVD): {word_svd_train.shape[1]}")
print(f"  - Char TF-IDF (SVD): {char_svd_train.shape[1]}")
print(f"  - TOTAL: {train_text_features_enhanced.shape[1]}")

Combining all enhanced text features...

✓ Enhanced text features shape:
  Train: (75000, 806)
  Test: (75000, 806)

Feature breakdown:
  - Numeric features: 12
  - Brand/quality features: 10
  - Sentence embeddings: 384
  - Word TF-IDF (SVD): 300
  - Char TF-IDF (SVD): 100
  - TOTAL: 806


## 6. Save Enhanced Features

In [ ]:
# Save the combined features
np.save(FEATURES_DIR / 'text_features_enhanced_train.npy', train_text_features_enhanced)
np.save(FEATURES_DIR / 'text_features_enhanced_test.npy', test_text_features_enhanced)

# Save individual components for flexibility
np.save(FEATURES_DIR / 'numeric_features_train.npy', train_numeric_features.values)
np.save(FEATURES_DIR / 'numeric_features_test.npy', test_numeric_features.values)

np.save(FEATURES_DIR / 'brand_features_train.npy', train_brand_features.values)
np.save(FEATURES_DIR / 'brand_features_test.npy', test_brand_features.values)

np.save(FEATURES_DIR / 'tfidf_enhanced_train.npy', train_tfidf_combined)
np.save(FEATURES_DIR / 'tfidf_enhanced_test.npy', test_tfidf_combined)

# Save vectorizers
with open(FEATURES_DIR / 'word_vectorizer_enhanced.pkl', 'wb') as f:
    pickle.dump(word_vectorizer, f)

with open(FEATURES_DIR / 'char_vectorizer_enhanced.pkl', 'wb') as f:
    pickle.dump(char_vectorizer, f)

with open(FEATURES_DIR / 'svd_word_enhanced.pkl', 'wb') as f:
    pickle.dump(svd_word, f)

with open(FEATURES_DIR / 'svd_char_enhanced.pkl', 'wb') as f:
    pickle.dump(svd_char, f)

# Save metadata
metadata = {
    'total_features': train_text_features_enhanced.shape[1],
    'numeric_features': train_numeric_features.shape[1],
    'brand_features': train_brand_features.shape[1],
    'sentence_embeddings': train_embeddings.shape[1] if SENTENCE_TRANSFORMER_AVAILABLE else 0,
    'word_tfidf_svd': word_svd_train.shape[1],
    'char_tfidf_svd': char_svd_train.shape[1],
    'word_variance_explained': float(svd_word.explained_variance_ratio_.sum()),
    'char_variance_explained': float(svd_char.explained_variance_ratio_.sum()),
    'sentence_transformer_model': MODEL_NAME if SENTENCE_TRANSFORMER_AVAILABLE else None,
}

with open(FEATURES_DIR / 'text_features_enhanced_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)